# Montezuma JupyterHub Setup

Run this notebook top-to-bottom once to set up the environment.  
After setup, use the **terminal** (File → New → Terminal) to run training jobs.

**Recommended profile:** `[GPU] Small (V100)` — 16 GB VRAM, 32 GB RAM, 8 CPUs.

## 1 · Verify the environment

In [ ]:
import sys, torch

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
    print("VRAM   :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

Expected output:
```
Python : 3.9.x ...
PyTorch: 2.x.x+cu...
CUDA   : True
GPU    : Tesla V100-SXM2-16GB
VRAM   : 16.0 GB
```
If CUDA is False, you selected a CPU-only profile — restart JupyterHub and pick `[GPU] Small (V100)`.

## 2 · Clone the repository

In [ ]:
import os, pathlib

REPO_URL = "https://github.com/YOUR_USERNAME/montezuma.git"  # ← update this
PROJECT_DIR = pathlib.Path.home() / "montezuma"

if PROJECT_DIR.exists():
    print("Repository already cloned — pulling latest changes.")
    os.system(f"git -C {PROJECT_DIR} pull")
else:
    os.system(f"git clone {REPO_URL} {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

## 3 · Install missing packages

PyTorch, TensorFlow, CUDA, numpy, tensorboard, and pillow are pre-installed on all GPU profiles.  
We only need the Atari / RL-specific packages.

In [ ]:
!pip install -q \
    "gymnasium==1.1.1" \
    "ale-py==0.11.2" \
    "AutoROM==0.6.1" \
    "wandb==0.18.7" \
    "opencv-python-headless"

print("Done.")

## 4 · Download Atari ROMs

In [ ]:
!python -m AutoROM --accept-license

## 5 · Verify the full stack

In [ ]:
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)
env = gym.make("ALE/MontezumaRevenge-v5", frameskip=1)
obs, _ = env.reset()
env.close()

print("gymnasium :", gym.__version__)
print("ale-py    :", ale_py.__version__)
print("Obs shape :", obs.shape)   # expect (210, 160, 3)
print("All good — environment loads correctly.")

## 6 · Smoke test (quick training run)

Runs ~50 k steps to confirm end-to-end training works and shows the GPU SPS.
This takes about 2–3 minutes.

> **Note:** `--sync-envs` is required on JupyterHub. `AsyncVectorEnv` uses Python
> `multiprocessing`, which does not work reliably inside Jupyter/container environments
> (`RuntimeError: Numpy is not available`). `--sync-envs` falls back to the
> single-process `SyncVectorEnv`. Performance is slightly lower but correct.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [
        sys.executable, "src/agents/rnd.py",
        "--total-timesteps", "50000",
        "--num-envs", "8",
        "--obs-norm-init-steps", "2",
        "--checkpoint-interval", "999",   # no checkpoint during smoke test
        "--runs-dir", "/tmp/smoke-runs",
        "--sync-envs",                    # required on JupyterHub (multiprocessing limitation)
    ],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else "(no output)")
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

Expected: `Using device: cuda`, SPS > 1 000.

---

## 7 · Running actual training jobs

**Do not run long jobs from a notebook cell** — the kernel dies if your browser disconnects.

Instead, open a **terminal** (File → New → Terminal) and use `nohup` or `tmux`:

```bash
cd ~/montezuma
mkdir -p logs

# Option A — nohup (simplest, logs to a file)
nohup python src/agents/rnd.py \
    --total-timesteps 50000000 \
    --num-envs 8 \
    --seed 1 \
    --sync-envs \
    --track \
    --wandb-project montezuma-thesis \
    > logs/rnd_seed1.out 2>&1 &
echo "PID: $!"

# Option B — tmux (recommended: lets you watch live output)
tmux new -s rnd
python src/agents/rnd.py \
    --total-timesteps 50000000 \
    --num-envs 8 \
    --seed 1 \
    --sync-envs \
    --track
# Ctrl+B then D  →  detach (job keeps running)
# tmux attach -t rnd  →  come back to it later
```

> **Why `--sync-envs`?** JupyterHub runs inside a container where Python's `multiprocessing`
> is unreliable. `--sync-envs` disables `AsyncVectorEnv` and uses the single-process
> `SyncVectorEnv` instead. Throughput is slightly lower but training is stable.

### Suggested settings for V100 on JupyterHub

| Flag | Value | Why |
|------|-------|-----|
| `--num-envs` | 8 | SyncVectorEnv is single-threaded; more envs don't help much |
| `--total-timesteps` | 5 000 000–50 000 000 | 1–10 h on V100; be considerate of shared resources |
| `--sync-envs` | always | Required on JupyterHub (see above) |
| `--track` | add flag | Monitor from W&B without keeping browser open |

### Monitor GPU usage
```bash
watch -n 2 nvidia-smi
```

### When done
**Shut down the GPU server** via File → Hub Control Panel → Stop My Server.  
This frees the V100 for other users.